In [1]:
import os 
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Olist-Ingestion") \
    .master("local[*]") \
    .getOrCreate()

spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
print("Spark session started ✅")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/17 14:21:12 WARN Utils: Your hostname, MacBook-Air-de-Sarah.local, resolves to a loopback address: 127.0.0.1; using 10.26.1.25 instead (on interface en0)
26/06/17 14:21:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/17 14:21:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark session started ✅


In [2]:
import os

base_path = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()

files = {
    "orders":      f"{base_path}/data/0_raw/olist_orders_dataset.csv",
    "customers":   f"{base_path}/data/0_raw/olist_customers_dataset.csv",
    "order_items": f"{base_path}/data/0_raw/olist_order_items_dataset.csv",
    "payments":    f"{base_path}/data/0_raw/olist_order_payments_dataset.csv",
    "products":    f"{base_path}/data/0_raw/olist_products_dataset.csv",
    "sellers":     f"{base_path}/data/0_raw/olist_sellers_dataset.csv",
    "reviews":     f"{base_path}/data/0_raw/olist_order_reviews_dataset.csv",
    "geolocation": f"{base_path}/data/0_raw/olist_geolocation_dataset.csv",
    "translation": f"{base_path}/data/0_raw/product_category_name_translation.csv",
}

In [3]:
dataframes = {}
for name, path in files.items():
    df = spark.read.csv(path, header=True, inferSchema=True, multiLine=True, escape='"')
    dataframes[name] = df
    print(f"{name} : {df.count()} rows")


orders : 99441 rows
customers : 99441 rows
order_items : 112650 rows
payments : 103886 rows
products : 32951 rows
sellers : 3095 rows
reviews : 99224 rows
geolocation : 1000163 rows
translation : 71 rows


In [4]:
df = spark.read.csv(
    path,
    header=True,
    inferSchema=True,
    multiLine=True,
    escape='"'
)
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

In [5]:
dataframes = {}
for name, path in files.items():
    df = spark.read.csv(path, header=True, inferSchema=True)
    dataframes[name] = df
    print(f"{name} : {df.count()} rows")
    

orders : 99441 rows
customers : 99441 rows
order_items : 112650 rows
payments : 103886 rows
products : 32951 rows
sellers : 3095 rows
reviews : 104162 rows
geolocation : 1000163 rows
translation : 71 rows


In [6]:
for name, df in dataframes.items():
    print(f"\n{'='*50}")
    print(f"Table : {name}")
    print(f"Rows : {df.count()}")
    df.printSchema()


Table : orders
Rows : 99441
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)


Table : customers
Rows : 99441
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)


Table : order_items
Rows : 112650
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = 

In [7]:
for name, df in dataframes.items():
    df.write.mode("overwrite").parquet(f"{base_path}/data/1_bronze/{name}")
    print(f"{name} saved to bronze ✅")

orders saved to bronze ✅
customers saved to bronze ✅
order_items saved to bronze ✅
payments saved to bronze ✅
products saved to bronze ✅
sellers saved to bronze ✅
reviews saved to bronze ✅


26/06/17 14:21:38 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers
26/06/17 14:21:38 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 84,44% for 9 writers
26/06/17 14:21:38 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 76,00% for 10 writers
26/06/17 14:21:38 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 84,44% for 9 writers


geolocation saved to bronze ✅
translation saved to bronze ✅


26/06/17 14:21:38 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers


In [9]:

table_names = list(dataframes.keys())

for i in range(len(table_names)):
    for j in range(i + 1, len(table_names)):
        name_a = table_names[i]
        name_b = table_names[j]
        cols_a = set(dataframes[name_a].columns)
        cols_b = set(dataframes[name_b].columns)
        common = cols_a & cols_b
        if common:
            print(f"{name_a} ↔ {name_b} : {common}")

orders ↔ customers : {'customer_id'}
orders ↔ order_items : {'order_id'}
orders ↔ payments : {'order_id'}
orders ↔ reviews : {'order_id'}
order_items ↔ payments : {'order_id'}
order_items ↔ products : {'product_id'}
order_items ↔ sellers : {'seller_id'}
order_items ↔ reviews : {'order_id'}
payments ↔ reviews : {'order_id'}
products ↔ translation : {'product_category_name'}
